[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cayley_klein/cayley_klein.ipynb)

# Cayley–Klein Geometry

Cayley–Klein geometry builds distances and angles out of a single conic. Projective geometry knows only which points lie on which lines; choose one conic, the absolute, and measure everything against it. With the unit circle as the absolute, the inside of the disk becomes the hyperbolic plane, the Beltrami–Klein model: straight lines stay straight, but distances grow without bound toward the rim and a triangle's angles sum to less than a half turn. Flip one sign of the conic and the same constructions do elliptic geometry instead.

The absolute is a polarity, a map from points to their polar lines, `Line <- Point`, built as a sum of dyads of the coordinate lines, `mv.x * (mv.x & Point) + ...`. Its inverse takes lines to their poles. Every metric notion comes out of these two maps and joins and meets: the invariant of two points, their pairing through the absolute over the roots of their self-pairings, is the hyperbolic cosine of their distance; the perpendiculars to a line all pass through its pole; reflection across a line is the harmonic homology about its pole; and a circle, the points at a fixed distance from a centre, is itself a polarity, a dyad of the centre's polar less a multiple of the absolute.

In the notation of projective geometry the absolute reads as a symmetric matrix $C$, Cayley's distance as $\cosh d = \frac{\mathbf p^{\mathsf T} C \mathbf q}{\sqrt{(\mathbf p^{\mathsf T} C \mathbf p)(\mathbf q^{\mathsf T} C \mathbf q)}}$, and the pole of a line $\mathbf l$ as $C^{-1}\mathbf l$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor
from examples.animation import save_figure
from examples.quadrics.cayley_klein import render
from examples.quadrics.cayley_klein.core import Line, Point, Pole, Polarity, Scalar, invariant, mv

np.set_printoptions(precision=4, suppress=True)

## 1. The absolute

The unit circle, `x**2 + y**2 - w**2 == 0`, as a polarity: each coordinate line paired with a point is a rank-one map from points to lines, and their signed sum is the absolute.

In [ ]:
absolute: Polarity = mv.x * (mv.x & Point) + mv.y * (mv.y & Point) - mv.w * (mv.w & Point)

In [ ]:
print("the absolute, a map from points to lines:\n", absolute.coefficient_table())

## 2. A triangle

The triangle's sides are the joins of consecutive vertices, given a side by the triangle's winding so that the absolute reads them; its angles are the invariants of the sides through the inverse absolute, its side lengths the invariants of the vertices, and by Gauss–Bonnet its area is its angle defect, a half turn less the angle sum.

In [ ]:
vertices = mv.point([[0, 0], [0.65, 0], [0.2, 0.55]])                          # [3] Point

In [ ]:
def triangle(absolute: Polarity, vertices: Point):
    """The sides, angles, side lengths and area of a triangle, all from the absolute."""
    dual_absolute: Pole = absolute.inverse()

    # The invariant of two points is (a & absolute(b)) / ((a & absolute(a)) * (b & absolute(b))).square_root(),
    # and Cayley's distance is its arccosh. The invariant of two lines is the same with the dual absolute,
    # and the angle is its arccos. Both stay inside the algebra until the very last step.
    distance_pairing = -(Point & absolute(Point))                         # [] Scalar <- (Point, Point)
    angle_pairing = Line & dual_absolute(Line)                            # [] Scalar <- (Line, Line)

    # A triangle: its sides are joins of consecutive vertices, batched, and the angle at
    # each vertex is between the two sides leaving it. Gauss-Bonnet gives the area as the
    # angle defect.
    neighbours = vertices[[1, 2, 0]]
    # The triangle's winding, an intrinsic scalar: the joins leaving each vertex times it are lines with a
    # side, which the absolute reads, and one factor on both lines of an angle leaves the angle as it is.
    winding = vertices[0] & vertices[1] & vertices[2]                      # [] Scalar
    to_next = (vertices & neighbours) * winding                            # [3] Line
    to_prev = (vertices & vertices[[2, 0, 1]]) * winding                   # [3] Line
    angles = invariant(angle_pairing, to_next, to_prev).clip(-1.0, 1.0).arccos()
    lengths = invariant(distance_pairing, vertices, neighbours).clip(1.0, np.inf).arccosh()
    area = np.pi - angles.sum()
    return to_next, angles, lengths, area


sides, angles, lengths, area = triangle(absolute, vertices)                    # [3] Line, [3] Scalar, [3] Scalar, [] Scalar

In [ ]:
print("angles:", angles.to_array(), "sum:", angles.sum().to_array())
print("side lengths:", lengths.to_array(), "area:", area.to_array())

## 3. A perpendicular and a reflection

Every line perpendicular to a side passes through the side's pole, so the perpendicular from a point is its join with the pole, and the foot is where that meets the side. Reflection across the side moves the point along the perpendicular to the other side of the foot, at the same distance.

In [ ]:
probe = mv.point([-0.15, 0.15])                                                # [] Point

In [ ]:
def perpendicular(absolute: Polarity, side: Line, point: Point):
    """The pole of a line, the perpendicular to it from a point, its foot, and the point reflected across
    the line."""
    # Every line perpendicular to a side passes through its pole, so the perpendicular from the point is
    # their join and the foot is its meet with the side. Reflection across the side is the harmonic
    # homology centred on the pole, and it preserves the distance to the foot.
    pole = absolute.solve(side)                                            # [] Point
    normal = point & pole                                                  # [] Line
    foot = side ^ normal                                                   # [] Point
    reflected = point - pole * (2.0 * (point & side) / (pole & side))      # [] Point
    return pole, normal, foot, reflected


pole, normal, foot, reflected = perpendicular(absolute, sides[1], probe)       # [] Point, Line, Point, Point
distance_pairing = -(Point & absolute(Point))                                  # [] Scalar <- (Point, Point)

In [ ]:
print("distances to the foot, from the point and from its reflection:",
      invariant(distance_pairing, probe, foot).arccosh().to_array(), invariant(distance_pairing, reflected, foot).arccosh().to_array())

## 4. Circles

The points at a fixed distance from a centre are those whose invariant with it is the hyperbolic cosine of the distance; clearing the square root makes that a quadric in the point, a polarity of its own. Circles of equal steps of radius crowd toward the rim, where each step covers less of the disk. Left, the triangle, the perpendicular from the point, its foot, the reflected point and the side's pole, inside the absolute; right, the circles about two centres.

In [ ]:
centres = mv.point([[0, 0], [0.45, 0.25]])                                     # [centres] Point
radii = np.array([0.25, 0.55, 0.9, 1.3])

In [ ]:
def circles(absolute: Polarity, centres: Point, radii: np.ndarray) -> Polarity:
    """Circles of the given radii about the given centres, as quadrics: one per centre and radius."""
    # Fixing the distance to a centre fixes the invariant, and clearing the square root
    # turns that into a quadric in P: the dyad of the centre's polar minus np.cosh(radii) ** 2
    # times the centre's self-invariant times the absolute. One expression, batched over the
    # radii, drawn as the level set P & circle(P) == 0.
    centre = centres[:, None]
    polar = absolute(centre)                                               # [centres, 1] Line
    return polar * (polar & Point) - (centre & polar) * np.cosh(radii) ** 2 * absolute   # [centres, radii] Line <- Point


rings = circles(absolute, centres, radii)                                      # [centres, radii] Line <- Point
save_figure(render.draw_hyperbolic_plane(absolute, vertices, sides, angles, area, probe, foot, normal, reflected, pole, rings, centres),
            "cayley_klein");

In [ ]:
# checks
# A hyperbolic angle defect; the perpendicular's pole, the meet of the polars of the two points it joins, lies on
# the side; and the point and its reflection are equally far from the foot.
assert area.to_array() > 0.0
np.testing.assert_allclose(((absolute(probe) ^ absolute(pole)) & sides[1]).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(invariant(distance_pairing, probe, foot).to_array(),
                           invariant(distance_pairing, reflected, foot).to_array(), atol=1e-12)